In [1]:
import os

# used for configuring biogeme use of GPU, unused
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Destination-choice + stay/move model via Larch

In [2]:
import os
import sys
from datetime import UTC, datetime
from pathlib import Path

import larch as lx
import numpy as np
import pandas as pd
from larch import PX

sys.path.insert(0, os.path.abspath(".."))
from lib import model_spec as lm
from lib import modeling_util as lut
from lib import io as lio


In [3]:
num_alternatives = 50
unixtime = int(datetime.now(UTC).timestamp())
path = "../data/estdata_10_2018_50.parquet"
data_file = Path(path).stem

### Read data

In [4]:
df_train = lio.read_estdata(
    path,
    num_alternatives,
)
print(df_train.shape)

(253090, 1114)


/workspace/migration/lib/io.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["person_id"] = np.arange(len(df))
/workspace/migration/lib/io.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["ALT_CHOICE"] = 0


In [5]:
# need to have the sentinel values be different so that SAME_CBSA works correctly
df_train["NAME_NUM.ORIG"].min(), df_train["ALT1_CBSA"].min()

(np.int64(-2), np.float32(-1.0))

### Look at collinearity in requested columns

In [6]:
# stack all alternatives into long format
frames = []
for i in range(1, num_alternatives + 1):
    cols = {
        f"ALT{i}_{v}": v
        for v in lm.required_alt_suffixes()
        if f"ALT{i}_{v}" in df_train.columns
    }
    frames.append(df_train[list(cols)].rename(columns=cols))

long = pd.concat(frames, ignore_index=True)

# add the transformed versions you actually use in the model
long["log_DIST"] = np.log(long["DIST"] + 1)
long["log_TOT_POP"] = np.log(long["TOT_POP"])

corr = long.corr()

In [7]:
c = corr.abs()
pairs = (
    c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print(pairs[pairs > 0.3])

TOT_POP                   log_TOT_POP                 0.992355
DIST                      log_DIST                    0.881614
MED_RENT_K                MED_HOUSE_VAL_100k          0.836388
                          MED_EARNINGS_K              0.721018
MED_EARNINGS_K            MED_HOUSE_VAL_100k          0.680974
MED_TRAVEL_TIME           TYPE                        0.611333
MED_RENT_K                FOREIGN_BORN_PROP           0.600982
                          TYPE                        0.589543
LF_PARTCP_RATE            HOUSE_VACANCY_PROP          0.568650
MED_RENT_K                MED_TRAVEL_TIME             0.553458
MED_TRAVEL_TIME           FOREIGN_BORN_PROP           0.541512
TYPE                      FOREIGN_BORN_PROP           0.528617
FOREIGN_BORN_PROP         MED_HOUSE_VAL_100k          0.527457
LF_PARTCP_RATE            MED_EARNINGS_K              0.503493
CBSA                      JAN_AVG_TEMP_C              0.498383
TYPE                      HOUSE_VACANCY_PROP          0

In [8]:
orig_vars = [v for v in lm.required_individual_columns() if v in df_train.columns]  # skip any missing
corr = df_train[orig_vars].corr()

c = corr.abs()
pairs = (
    c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print(pairs[pairs > 0.3].to_string())

ORIGIN_STATE                                                CHOSEN                                                        0.974723
FOREIGN                                                     POBP                                                          0.903096
Median house cost in hundreds of thousands of dollars.ORIG  Median gross rent in thousands of dollars.ORIG                0.889116
                                                            Proportion of people AAPI.ORIG                                0.840296
Proportion of people 18-34.ORIG                             Proportion of people in college.ORIG                          0.800195
Median gross rent in thousands of dollars.ORIG              Proportion of people AAPI.ORIG                                0.792994
EDU_HIGH                                                    EDU_BACHELORS                                                 0.786110
Median gross rent in thousands of dollars.ORIG              Proportion foreign born

### Reshape to long format, build the Larch dataset

`lib.util.build_long_data` builds the long `(person_id, alt)` table shared by this notebook and
`modeling_torch_choice.ipynb`: `alt=0` is staying, `alt=1..num_alternatives` are the move alternatives, with the
stay/move-context values for shared coefficients (e.g. `proportion_same_age_18_34`) written under the
same column name so they tie to one coefficient downstream, and `log_pop_offset` (destination
population on move rows, origin population on the stay row) left as an un-parameterized term. 

The returned `long_df` is already sorted by `(person_id, alt)`; `Dataset.construct.from_idca` takes it
directly (indexed by `(caseid, altid)`) -- no manual reshape into arrays needed, unlike the torch-choice
port.


In [9]:
long_df, STAY_ONLY_TERMS, SHARED_TERMS, MOVE_ONLY_TERMS = lut.build_long_data(
    df_train, num_alternatives
)
varnames = STAY_ONLY_TERMS + SHARED_TERMS + MOVE_ONLY_TERMS

long_df.set_index(["person_id", "alt"], inplace=True)

In [10]:
lut.print_utility_formula(long_df.reset_index()[["person_id", "alt", "choice", "log_pop_offset"] + varnames])

Stay utility:
    Beta(stay)*Variable(stay)
    + Beta(stay_age_18_22)*Variable(stay_age_18_22)
    + Beta(stay_age_23_29)*Variable(stay_age_23_29)
    + Beta(stay_age_30_39)*Variable(stay_age_30_39)
    + Beta(stay_age_40_49)*Variable(stay_age_40_49)
    + Beta(stay_age_50_64)*Variable(stay_age_50_64)
    + Beta(stay_child_under_6)*Variable(stay_child_under_6)
    + Beta(stay_child_6_to_17)*Variable(stay_child_6_to_17)
    + Beta(stay_married_more_than_year)*Variable(stay_married_more_than_year)
    + Beta(stay_married_less_than_year)*Variable(stay_married_less_than_year)
    + Beta(stay_recently_divorced_or_widowed)*Variable(stay_recently_divorced_or_widowed)
    + Beta(stay_2work_mar)*Variable(stay_2work_mar)
    + Beta(stay_single_parent)*Variable(stay_single_parent)
    + Beta(stay_edu_college)*Variable(stay_edu_college)
    + Beta(stay_edu_high)*Variable(stay_edu_high)
    + Beta(stay_in_college)*Variable(stay_in_college)
    + Beta(stay_foreign)*Variable(stay_foreign)
    + Beta

In [11]:
ds = lx.Dataset.construct.from_idca(
    long_df[["choice", "log_pop_offset"] + varnames], crack=False
)
ds

<xarray.Dataset> Size: 3GB
Dimensions:                                       (person_id: 253090, alt: 51)
Coordinates:
  * person_id                                     (person_id) int64 2MB 0 ......
  * alt                                           (alt) int64 408B 0 1 ... 49 50
Data variables: (12/60)
    choice                                        (person_id, alt) int64 103MB ...
    log_pop_offset                                (person_id, alt) float32 52MB ...
    stay                                          (person_id, alt) float32 52MB ...
    stay_age_18_22                                (person_id, alt) float32 52MB ...
    stay_age_23_29                                (person_id, alt) float32 52MB ...
    stay_age_30_39                                (person_id, alt) float32 52MB ...
    ...                                            ...
    destchoice_samecbsa                           (person_id, alt) float32 52MB ...
    destchoice_samestate                          (person_id, alt) float32 52MB ...
    destchoice_birthstate                         (person_id, alt) float32 52MB ...
    destchoice_T34                                (person_id, alt) float32 52MB ...
    destchoice_metro                              (person_id, alt) float32 52MB ...
    destchoice_same_cbsa_type                     (person_id, alt) float32 52MB ...
Attributes:
    _caseid_:  person_id
    _altid_:   alt

### Setting up the model

One `P(name) * X(name)` term per `varnames` entry via the `PX` shorthand, summed on a plain local
variable and assigned to `m.utility_ca` once at the end -- **not** built with `m.utility_ca += ...` in
a loop, which silently discards everything but the last term (see intro cell). No separate ASC/intercept
term is added -- `stay` in `STAY_ONLY_TERMS` is already an explicit ASC for staying, matching
`fit_intercept=False` in the torch-choice port / Biogeme not adding an implicit ASC of its own.

`log_pop_offset` gets a coefficient too, then `m.lock_value("log_pop_offset", 1)` pins it at exactly 1
(`holdfast`), reproducing Biogeme's bare `log(Variable(...))` calls (and xlogit's `addit=`) without
needing a custom subclass the way the torch-choice port did.


In [ ]:
m = lx.Model(ds)
m.title = f"us_mnl_{data_file}_{unixtime}"
m.compute_engine = "numba"

# all alternatives have the same utility function
# stay-specific columns have their values zeroed out for destination alternatives and vice versa
# PX represents a column multiplied by a coefficient that will be estimated
total_utility = PX(varnames[0])
for name in varnames[1:]:
    total_utility = total_utility + PX(name)
total_utility = total_utility + PX("log_pop_offset")
m.utility_ca = total_utility

m.choice_ca_var = "choice"
# all alternatives are available for everyone
# no availability_ca_var needed.

# fix the size term coefficient, it is a constant
m.lock_value("log_pop_offset", 1)

m.ordering = [
    ("Stay", "stay.*"),
    ("Destination-only", "destchoice.*"),
    ("Offset", "log_pop_offset"),
]


In [13]:
# NOTE: nesting showed that mu_move tended to go towards 1, indicating that nesting is not necessary

# # optional cell: turns on the nested structure

# # nested logit: alt=0 (stay) stays a direct root child (== a degenerate nest fixed at 1.0);
# # alts 1..num_alternatives go under a "Move" nest with an estimated logsum coefficient.
# m.graph.new_node(
#     parameter="mu_move",
#     children=list(range(1, num_alternatives + 1)),
#     name="Move",
# )
# m.set_value("mu_move", value=0.5, initvalue=0.5, minimum=0.001, maximum=1.0)

# m.ordering = [
#     ("Stay", "stay.*"),
#     ("Shared", "proportion.*|median_.*|unemp_rate|vacancy_rate"),
#     ("Destination-only", "destchoice.*"),
#     ("Nesting", "mu_.*"),
#     ("Offset", "log_pop_offset"),
# ]
# m.title = f"us_nested_{data_file}_{unixtime}"


### Fitting

In [14]:
print("null log-likelihood:", m.loglike())


null log-likelihood: -726809.5990740261


In [15]:
result = m.maximize_loglike(method="BHHH")
result


┣          loglike: np.float64(-117926.72968127244)
┣                x: amenities_est_per_capita                       -83.002638
┃                   amenities_est_per_capita_18_34                  38.348523
┃                   amenities_est_per_capita_35_64                  17.134621
┃                   destchoice_T34                                  -0.453540
┃                   destchoice_birthstate                            0.305226
┃                   destchoice_logdist                              -0.950069
┃                   destchoice_metro                                -0.173411
┃                   destchoice_same_cbsa_type                        0.278746
┃                   destchoice_samecbsa                              1.505124
┃                   destchoice_samestate                             2.061723
┃                   jan_avg_temp_c                                   0.015222
┃                   lf_prop_if_in_lf                                -0.756579
┃                   log_pop_offset                                   1.000000
┃                   med_earnings_10k                                 0.063823
┃                   med_house_val_100k                               0.006028
┃                   med_rent_1k                                     -0.594210
┃                   median_travel_time                              -0.033063
┃                   proportion_also_latino                           0.831449
┃                   proportion_also_mil                             26.065921
┃                   proportion_college_if_in_college                10.798927
┃                   proportion_foreign_if_foreign                    1.640479
┃                   proportion_hh_with_children_if_have_children     3.852871
┃                   proportion_same_age_18_34                        2.995049
┃                   proportion_same_age_35_64                        2.391261
┃                   proportion_same_age_65_plus                      3.548154
┃                   proportion_same_naics_agr_ext                    6.355321
┃                   proportion_same_naics_goods_trade                1.550587
┃                   proportion_same_naics_govt                       2.477420
┃                   proportion_same_naics_high_ed                    0.638054
┃                   proportion_same_race_aapi                        3.706965
┃                   proportion_same_race_black                       2.040306
┃                   proportion_same_race_indian                      4.707415
┃                   proportion_same_race_white                       2.192551
┃                   stay                                            -6.535251
┃                   stay_2work_mar                                   0.725258
┃                   stay_T34                                         0.240057
┃                   stay_age_18_22                                  -1.851425
┃                   stay_age_23_29                                  -1.788944
┃                   stay_age_30_39                                  -1.414637
┃                   stay_age_40_49                                  -0.970176
┃                   stay_age_50_64                                  -0.442188
┃                   stay_child_6_to_17                               0.416352
┃                   stay_child_under_6                              -0.213914
┃                   stay_edu_college                                -0.110397
┃                   stay_edu_high                                    0.052761
┃                   stay_foreign                                    -0.257239
┃                   stay_in_college                                 -0.109544
┃                   stay_married_less_than_year                     -0.713471
┃                   stay_married_more_than_year                      0.496023
┃                   stay_metro                                       0.142150
┃                   stay_mil                    

In [16]:
m.calculate_parameter_covariance()
m.parameter_summary()


/tmp/ipykernel_7901/1976790631.py:1: PossibleOverspecification: Model is possibly over-specified (hessian is nearly singular).
  m.calculate_parameter_covariance()


In [17]:
print(long_df[["amenities_est_per_capita", "med_rent_1k", "median_travel_time",
            "destchoice_metro", "destchoice_T34"]].corr().round(2))

                          amenities_est_per_capita  med_rent_1k  \
amenities_est_per_capita                      1.00         0.22   
med_rent_1k                                   0.22         1.00   
median_travel_time                           -0.02         0.56   
destchoice_metro                             -0.09        -0.25   
destchoice_T34                                0.07         0.54   

                          median_travel_time  destchoice_metro  destchoice_T34  
amenities_est_per_capita               -0.02             -0.09            0.07  
med_rent_1k                             0.56             -0.25            0.54  
median_travel_time                      1.00             -0.38            0.62  
destchoice_metro                       -0.38              1.00           -0.70  
destchoice_T34                          0.62             -0.70            1.00  


In [18]:
report = lx.Reporter(title=m.title)
report << "# Parameter Summary" << m.parameter_summary()
report << "# Estimation Statistics" << m.estimation_statistics()
report.save(
    f"results/{m.title}.html",
    overwrite=True,
    metadata=m.dumps(),
)
m.save(f"results/{m.title}_spec.yaml")